# 05 Smoke-проверка доступа pi_0 и ограничение обучения

        Этот раздел не торопит с запуском длительного обучения, а сначала проверяет доступ к PaliGemma и Hugging Face (gated), загрузку весов, загрузку данных и 1-step smoke. Полное обучение имеет смысл только после прохождения smoke.


## Важное исправление: старая панель из 30 seed — это не генерализация по положению

        `y_env2.py` раньше всегда задавал `np.random.seed(seed=0)` независимо от переданного seed окружения. Старая панель `1010-1039` меняла только сэмплирование политики Pi0, а положения чашки и тарелки оставались фиксированными. Эти результаты всё ещё можно использовать для анализа временной структуры действий scaffold, но их нельзя считать доказательством пространственной генерализации. После исправления контроль raw/head в фиксированном окружении даёт `0/12 -> 6/12`, а gate по случайным окружениям — только `1/4`; следующим шагом нужно пересобрать данные с несколькими положениями.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: шаблон команды проверки доступа


In [ ]:
print("""
cd "$PROJECT_ROOT"
HF_TOKEN_STDIN=1 REQUIRE_PROXY=1 ./install_hf_token_for_pi0.sh
""")


Эта форма команды подчёркивает две вещи: token не записывается в Notebook, а скрипт сначала проверяет `whoami`, PaliGemma и `lerobot/pi0`, и только после успешной проверки всего сохраняет результат.


## Checkpoint 2: что доказывает 1-step smoke


In [ ]:
rows = [
    ("Доступ к gated-модели", "можно прочитать конфигурацию, нужную для PaliGemma / pi0"),
    ("Загрузка данных", "читаются набор данных LeRobot и ключи observation/action"),
    ("Построение policy", "policy pi0 инициализируется"),
    ("Один шаг обучения", "forward/backward/шаг оптимизатора выполняются"),
    ("Запись checkpoint", "в выходном каталоге появляется smoke checkpoint"),
]
md_table(["Проверка", "Что означает прохождение"], rows)


1-step smoke не доказывает сходимость политики и не означает её успешность. Это просто порог перед полным обучением.


## Checkpoint 3: шаблон команды обучения


In [ ]:
print("""
cd "$PROJECT_ROOT"
RUN_SMOKE=1 RUN_FULL_TRAIN=1 PI0_STEPS=20000 PI0_BATCH_SIZE=4 \\
  ./run_pi0_train_eval_after_hf_ready.sh
""")


Полное обучение лучше запускать в терминале, tmux или фоновом скрипте. Notebook отвечает за запись параметров, объяснение ограничений и чтение summary после обучения.


## Checkpoint 4: диаграмма этапов


In [ ]:
show_asset("model_status_summary.png", width=900)


## Checkpoint 5: диагностика финишера второй половины с `tcp_to_plate`


Сырой pi_0 пока нельзя записывать как воспроизведение успеха. Более ценная итерация диагностики — расширить state финишера второй половины с 19 до 22 измерений, добавив `tcp_to_plate = tcp_link_xyz - plate_xyz`. Это явно сообщает модели направление от центра тарелки к TCP и не даёт ей неявно угадывать цель только по изображению и phase.

        Этот эксперимент по-прежнему является диагностическим scaffold: префикс предоставляется oracle, schedule выровнен по `move_preplace`, финишер использует 22D state. Последующая проверка на 30 seed показала, что короткий schedule `move_preplace` преждевременно опускает и отпускает объект; после принудительного длинного шаблона schedule улучшенный scaffold стабильно проходит 30 seed сэмплирования политики (фактически окружение фиксировано на seed 0). Это доказывает, что узкое место найдено и устранено, но не означает, что raw pi_0 уже успешен сквозным образом.

        Следующий шаг — разбирать scaffold. Первым заменили жёсткое правило захвата: logistic-head только по phase предсказывает открытие и закрытие захвата, EEF/рука по-прежнему выводится финишером 22D. Full-state gripper head даёт точность на обучающей выборке `100%`, но в замкнутом цикле проваливается; phase-only head видит только `timestamp + phase_index_norm + phase_onehot11`, и на полной выборке seed сэмплирования политики `1010-1039` (фактически окружение фиксировано на seed 0) сохраняет strict `30/30`. Этот контроль показывает: для head по коротким событиям на малых данных входы нужно ограничивать и не подавать туда все состояния руки, которые в замкнутом цикле дрейфуют.

        Второй блок scaffold сначала заменили адаптивным gate с фиксированным порогом вместо принудительного длинного шаблона `move_preplace`: на полной выборке seed `1010-1039` (фактически окружение фиксировано на seed 0) результат всё ещё strict `30/30`, но часть прогонов опирается на страховку `max_steps=180`. Дальше обучили logistic transition head. Полная первая версия даёт точность на обучающей выборке около `99.27%`, но в замкнутом цикле seed `1010` преждевременно проваливается; вторая версия сохраняет только `tcp_to_plate_xy + local_step_norm`, точность на обучающей выборке около `95.01%`, однако на полной выборке `1010-1039` даёт strict `30/30`, legacy `30/30`. В 30 прогонах `29/30` переключений вызваны transition head, и только seed `1021` дошёл до страховки по максимальному числу шагов. Этот контроль показывает: входы phase head на малых данных тоже нужно ограничивать; стабильные геометрические признаки важнее высокой точности на обучающей выборке.

        Третий уровень — разбор примитива контакта в первой половине. Naive all-head учит все переходы по хвосту phase: метрики на обучении выглядят неплохо, но smoke `1010/1011` падает до `0/2`, потому что `descend_to_close` срабатывает слишком рано, когда TCP ещё на `0.08 m` выше пола захвата. Исправленная версия переводит `pregrasp_to_descend` в геометрическую метку, добавляет в признаки относительные величины TCP до grasp / pregrasp / floor и при развёртывании включает `descend_floor_guard`. Полная выборка `1010-1039` даёт strict `30/30`, legacy `30/30`; ключевые переключения `pregrasp->descend`, `descend->close`, `close->lift` все `30/30` срабатывают по команде head, а floor guard заблокировал `342` высоких закрытий. Это всё ещё contact scaffold, а не сквозной успех raw pi_0.

        Четвёртый уровень — убираем хвост `dataset_schedule` и переходим к финишеру `dynamic_timed`. Первый полный прогон на 30 seed дал strict только `27/30`; провальные seed `1021/1031/1036` выглядели так, будто фиксированная пауза не подходит. После разбора кода корень оказался в глобальном переключателе `--tcpplate-prefix-target-state`: префикс нуждается в `tcp_to_target`, но dynamic finisher тоже его использовал, хотя при обучении финишер должен получать `tcp_to_plate`. Когда builder состояния стал учитывать стадию, на стадии префикса используется положение относительно цели, а на стадии финишера — относительно тарелки; три провальных seed при повторной проверке дали `3/3`, а прежняя версия в том же окружении на полной выборке `1010-1039` снова дала strict `30/30`, legacy `30/30`. Но при дальнейшей трассировке выяснилось, что при непрерывной оценке остаточная динамика MuJoCo (qvel / ctrl / free-joint) от предыдущего episode может загрязнять следующий seed: у seed `1035` начальная позиция однажды оказалась вне диапазона сэмплирования. Поэтому в evaluator добавлены `--fresh-env-per-episode` и `--hard-reset-sim-data`. Итоговый чистый протокол с `--hard-reset-sim-data` на полной выборке `1010-1039` снова даёт strict `30/30`, legacy `30/30`, красная чашка `18/18`, синяя `12/12`; средний `xy=0.0219 m`, максимальный `xy=0.0450 m`, минимальный lift `0.1093 m`, минимальный upright cos `0.9504`. Прежнее значение `0.0993 m` у seed `1036` больше похоже на артефакт оценки из-за остаточного состояния reset, поэтому его больше не считают пограничным образцом.


In [ ]:
rows = [
    ("raw full20 open-loop", "1/20 final, 3/20 ever", "raw policy всё ещё не проходит"),
    ("raw closed-loop strict", "0/20", "ближе всего к условиям развёртывания"),
    ("template-tail full20", "4/20", "фиксированный хвост спасает только часть образцов"),
    ("старая передача 19D finisher", "prefix 120/180/240/300: все 0/2", "цель второй половины всё ещё смещена к стороне чашки"),
    ("22D tcp_to_plate finisher", "strict 5/10, legacy 6/10", "state относительно тарелки — полезный сигнал, но это всё ещё scaffold"),
    ("22D + phase-scripted gripper", "strict 7/10, legacy 7/10", "тайминг захвата спасает часть провалов; оставшиеся — в основном место посадки красной чашки"),
    ("фиксированное окружение, 30 policy seeds, schedule 0..9", "strict 21/30, legacy 23/30", "все провалы связаны с коротким шаблоном move_preplace"),
    ("фиксированное окружение, 30 policy seeds, принудительный длинный schedule", "strict 30/30, legacy 30/30", "доказывает только стабильность scaffold для фиксированной сцены"),
    ("длинный schedule + phase-only learned gripper head", "strict 30/30, legacy 30/30", "первый блок scaffold заменён обучаемым head"),
    ("schedule 0..9 + adaptive move gate + learned gripper", "strict 30/30, legacy 30/30", "второй блок scaffold, инженерная версия: длинный episode0 больше не навязывается"),
    ("schedule 0..9 + xy-step transition head + learned gripper", "strict 30/30, legacy 30/30", "второй блок scaffold, обучаемая версия: 29/30 срабатываний по команде head"),
    ("policy prefix + contact transition heads + floor guard", "strict 30/30, legacy 30/30", "третий уровень без scaffold: ключевые переходы контакта срабатывают по команде head, но scaffold сохраняется"),
    ("stage-aware dynamic_timed finisher + hard-reset clean protocol", "strict 30/30, legacy 30/30", "четвёртый уровень без scaffold: prefix использует target, finisher — plate; mean xy=0.0219 m, max xy=0.0450 m"),
]
md_table(["Критерий", "Результат", "Как понимать"], rows)


In [ ]:
print("""
cd "$PROJECT_ROOT"

# 1. Преобразуем данные 19D phase-state в данные 22D tcp_to_plate.
python code/pi0/build_lerobot_state_phase_tcpplate.py \\
  --src-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_eef_abs_10eps_v1 \\
  --src-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_eef_abs_10eps_v1 \\
  --dst-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dst-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --overwrite

# 2. Оценка 22D finisher. Внимание: эта команда требует уже существующий 22D checkpoint.
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1000,1001,1002,1003,1004,1005,1006,1007,1008,1009 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval/summary.json"

# 3. При проверке на большей выборке можно принудительно использовать длинный шаблон move_preplace.
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-force-schedule-episode 0 \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_long_schedule/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_long_schedule/summary.json"

# 4. Обучаем phase-only gripper head. Этот head заменяет только правило захвата и не меняет EEF/рук.
python code/pi0/train_tcpplate_gripper_head.py \\
  --dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --feature-mode phase \\
  --label-source action \\
  --output "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --summary-json "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg_summary.json"

# 5. Проверяем 30 seed сэмплирования политики с learned gripper head (окружение фактически фиксировано на seed 0).
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-force-schedule-episode 0 \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_gripper_head/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_gripper_head/summary.json"

# 6. Убираем принудительный длинный episode0 и используем adaptive gate для move_preplace.
# Эта версия использует schedule 0..9 с обычным повтором; если live TCP-to-plate ещё слишком далеко,
# удерживаем move_preplace, пока xy не станет достаточно малым или не будет достигнут консервативный предел max-step.
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-adaptive-move-preplace \\
  --tcpplate-adaptive-move-preplace-xy-threshold 0.05 \\
  --tcpplate-adaptive-move-preplace-min-steps 20 \\
  --tcpplate-adaptive-move-preplace-max-steps 180 \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_adaptive_gate/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_adaptive_gate/summary.json"

# 7. Обучаем xy-step transition head. Full-state transition head точнее на обучающей выборке,
# но её может сбивать корреляция с высотой и направлением; стабильная версия использует только tcp_to_plate_xy + local_step_norm.
python code/pi0/train_tcpplate_transition_head.py \\
  --dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --feature-mode xy_step \\
  --step-scale 180 \\
  --output "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg.npz" \\
  --summary-json "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg_summary.json"

# 8. Заменяем фиксированный пороговый gate обучаемым transition head; schedule 0..9 повторяется обычно.
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg.npz" \\
  --tcpplate-transition-head-threshold 0.5 \\
  --tcpplate-adaptive-move-preplace-min-steps 20 \\
  --tcpplate-adaptive-move-preplace-max-steps 180 \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_transition_head/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_transition_head/summary.json"

# 9. Обучаем target-relative contact transition heads.
# naive all-head выучит «стадия скоро закончится», а не «сейчас безопасно закрыть захват»;
# здесь используются метка геометрии pregrasp и признаки геометрии grasp/floor.
python code/pi0/train_tcptarget_contact_transition_head.py \\
  --dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --task-set all \\
  --early-label-mode pregrasp_geometry \\
  --feature-mode grasp_geom_count_step \\
  --step-scale 180 \\
  --output-npz "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step.npz" \\
  --summary-json "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step_summary.json"

# 10. Contact transition heads заменяют часть ручных переключений pregrasp/descend/close в первой половине.
# descend_floor_guard сохраняется, чтобы не закрыть захват раньше времени, когда TCP всё ещё заметно выше пола захвата.
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-prefix-target-state \\
  --tcpplate-contact-primitive guided_contact_lift \\
  --tcpplate-contact-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step.npz" \\
  --tcpplate-contact-transition-head-threshold 0.5 \\
  --tcpplate-contact-pregrasp-hold 999 \\
  --tcpplate-contact-descend-hold 999 \\
  --tcpplate-contact-transition-head-strict \\
  --tcpplate-contact-descend-floor-guard \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --tcpplate-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg.npz" \\
  --tcpplate-adaptive-move-preplace-min-steps 20 \\
  --tcpplate-adaptive-move-preplace-max-steps 180 \\
  --prefix-source policy \\
  --prefix-policy-path "$MODEL_ROOT/pi0_prefix_tcptarget/checkpoints/001000/pretrained_model" \\
  --prefix-dataset-repo-id datawhale_eai_pnp_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-dataset-root ./demo_data_pi0_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_contact_transition_head_unseen30/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_contact_transition_head_unseen30/summary.json"

# 11. Убираем хвост dataset_schedule и используем stage-aware dynamic_timed finisher.
# Ключевой момент: --tcpplate-prefix-target-state задаёт tcp_to_target только для prefix;
# stage-aware evaluator возвращает finisher к tcp_to_plate.
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-prefix-target-state \\
  --tcpplate-contact-primitive guided_contact_lift \\
  --tcpplate-contact-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step.npz" \\
  --tcpplate-contact-transition-head-threshold 0.5 \\
  --tcpplate-contact-pregrasp-hold 999 \\
  --tcpplate-contact-descend-hold 999 \\
  --tcpplate-contact-transition-head-strict \\
  --tcpplate-contact-descend-floor-guard \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source policy \\
  --prefix-policy-path "$MODEL_ROOT/pi0_prefix_tcptarget/checkpoints/001000/pretrained_model" \\
  --prefix-dataset-repo-id datawhale_eai_pnp_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-dataset-root ./demo_data_pi0_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-steps 180 \\
  --finisher-phase-state dynamic_timed \\
  --finisher-start-phase move_preplace \\
  --timed-phase-dwell-spec move_preplace:260,lower_to_plate:40,retreat:40 \\
  --hard-reset-sim-data \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_dynamic_timed_stageaware_unseen30/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_dynamic_timed_stageaware_unseen30/summary.json"
""")


Чтобы запустить контроль с scripted-захватом, добавьте в команду оценки `--finisher-scripted-gripper`. Этот флаг меняет только gripper, не трогая EEF/рука, и показывает, является ли тайминг открытия и закрытия захвата главным узким местом. `--tcpplate-force-schedule-episode 0` проверяет, устраняет ли длинный шаблон `move_preplace` преждевременное освобождение. `--tcpplate-gripper-head-path` загружает learned head в хук `--finisher-scripted-gripper` и заменяет ручное правило phase обучаемой малой моделью. `--tcpplate-adaptive-move-preplace` заменяет принудительный длинный шаблон live progress gate; `--tcpplate-transition-head-path` заменяет фиксированный пороговый gate logistic transition head; `--tcpplate-contact-transition-head-path` дополнительно заменяет часть переключений phase в примитиве контакта первой половины. Для версии `dynamic_timed` особенно важен stage-aware state: prefix может использовать `tcp_to_target`, а finisher обязан использовать `tcp_to_plate`. Для пакетной оценки рекомендуется `--hard-reset-sim-data`, чтобы сначала убрать остаточную динамику MuJoCo между episode; `--fresh-env-per-episode` чище, но при частом создании окружения может возникать случайная ошибка провайдера ассетов. Текущий stage-aware dynamic finisher в чистом протоколе hard-reset на 30 seed сэмплирования политики (окружение фактически фиксировано на seed 0) даёт strict `30/30`, но всё ещё опирается на target-relative contact scaffold, floor guard, learned gripper head и финишер второй половины.


Если state normalizer старого checkpoint имеет размерность 19, а новые данные — 22, появится `mean/std` shape mismatch. В диагностическом эксперименте можно скопировать checkpoint и удалить `normalize_inputs.buffer_observation_state.mean/std`, чтобы статистики 22D набора данных инициализировались заново; такую временную обработку не нужно записывать как архитектурное новшество модели.
